# Assessment Task 3 — Text Classification: Sarcasm Detection
**Saharsh Pathak | 2417371 | Herald College Kathmandu**

Binary text classification to detect sarcasm in news headlines.
Dataset: `sarcastic_headlines.csv` — 26,709 headlines labelled as sarcastic (1) or not (0).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import string
from collections import Counter

# NLP
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print('Libraries loaded')

## 1. Load & Explore Dataset

In [ ]:
# Load sarcastic headlines dataset
df = pd.read_csv('sarcastic_headlines.csv')
print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
print(df.head())
print(f'\nClass distribution:')
print(df['is_sarcastic'].value_counts())

In [ ]:
# Visualise class balance
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df['is_sarcastic'].value_counts()
axes[0].bar(['Not Sarcastic', 'Sarcastic'], counts.values, color=['#0EA5E9', '#EF4444'])
axes[0].set_title('Class Distribution')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 100, str(v), ha='center', fontweight='bold')

# Headline length distribution
df['headline_len'] = df['headline'].str.split().str.len()
axes[1].hist(df[df['is_sarcastic']==0]['headline_len'], bins=30, alpha=0.7, color='#0EA5E9', label='Not Sarcastic')
axes[1].hist(df[df['is_sarcastic']==1]['headline_len'], bins=30, alpha=0.7, color='#EF4444', label='Sarcastic')
axes[1].set_title('Headline Length Distribution')
axes[1].set_xlabel('Word Count')
axes[1].legend()

plt.tight_layout()
plt.show()

## 2. Text Preprocessing

In [ ]:
def preprocess_text(text):
    """Clean and normalise text for classification."""
    text = text.lower()                                    # lowercase
    text = re.sub(r'http\S+|www\S+', '', text)            # remove URLs
    text = re.sub(r'[^a-z\s]', '', text)                  # remove punctuation/numbers
    text = re.sub(r'\s+', ' ', text).strip()              # remove extra spaces
    return text

df['clean_headline'] = df['headline'].apply(preprocess_text)

print('Before:', df['headline'].iloc[0])
print('After: ', df['clean_headline'].iloc[0])
print(f'\nTotal samples: {len(df)}')

## 3. Feature Extraction — TF-IDF

In [ ]:
X = df['clean_headline']
y = df['is_sarcastic']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# TF-IDF vectorisation
tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),      # unigrams + bigrams
    stop_words='english',
    min_df=2
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print(f'Train: {X_train_tfidf.shape}')
print(f'Test:  {X_test_tfidf.shape}')
print(f'Vocabulary size: {len(tfidf.vocabulary_)}')

## 4. Model Training & Comparison

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, C=1.0, random_state=42),
    'Naive Bayes':         MultinomialNB(alpha=0.1),
    'Linear SVM':          LinearSVC(C=1.0, max_iter=2000, random_state=42)
}

results = {}
for name, model in models.items():
    model.fit(X_train_tfidf, y_train)
    y_pred = model.predict(X_test_tfidf)
    acc = accuracy_score(y_test, y_pred)
    results[name] = {'model': model, 'acc': acc, 'pred': y_pred}
    print(f'{name:25s}: {acc:.4f} ({acc*100:.2f}%)')

## 5. Best Model Evaluation

In [ ]:
best_name = max(results, key=lambda k: results[k]['acc'])
best = results[best_name]
print(f'Best Model: {best_name} ({best["acc"]:.4f})')
print('\nClassification Report:')
print(classification_report(y_test, best['pred'], target_names=['Not Sarcastic', 'Sarcastic']))

# Confusion matrix
cm = confusion_matrix(y_test, best['pred'])
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Not Sarcastic', 'Sarcastic'],
            yticklabels=['Not Sarcastic', 'Sarcastic'])
plt.title(f'Confusion Matrix — {best_name}')
plt.ylabel('Actual'); plt.xlabel('Predicted')
plt.tight_layout(); plt.show()

## 6. Model Comparison Chart

In [ ]:
names = list(results.keys())
accs = [results[n]['acc'] for n in names]

plt.figure(figsize=(9, 4))
bars = plt.bar(names, accs, color=['#0EA5E9', '#6366F1', '#10B981'], width=0.5)
plt.ylim(0.75, 1.0)
plt.title('Sarcasm Detection — Model Accuracy Comparison')
plt.ylabel('Accuracy')
for bar, acc in zip(bars, accs):
    plt.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003,
             f'{acc:.4f}', ha='center', fontweight='bold')
plt.tight_layout(); plt.show()

## 7. Top Sarcasm Indicators

In [ ]:
# Top features for Logistic Regression
lr = results['Logistic Regression']['model']
feature_names = tfidf.get_feature_names_out()
coefs = lr.coef_[0]

top_sarcastic = np.argsort(coefs)[-15:]
top_not_sarcastic = np.argsort(coefs)[:15]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].barh(range(15), coefs[top_sarcastic], color='#EF4444')
axes[0].set_yticks(range(15))
axes[0].set_yticklabels([feature_names[i] for i in top_sarcastic])
axes[0].set_title('Top Sarcasm Indicators')
axes[0].set_xlabel('Coefficient')

axes[1].barh(range(15), coefs[top_not_sarcastic], color='#0EA5E9')
axes[1].set_yticks(range(15))
axes[1].set_yticklabels([feature_names[i] for i in top_not_sarcastic])
axes[1].set_title('Top Non-Sarcasm Indicators')
axes[1].set_xlabel('Coefficient')

plt.tight_layout(); plt.show()

## 8. Test on Custom Headlines

In [ ]:
test_headlines = [
    "Scientists discover that eating chocolate every day makes you smarter",
    "Government announces new policy to reduce carbon emissions",
    "Man who hates mornings somehow survives another Monday",
    "New study confirms that exercise is good for health",
    "Area man absolutely thrilled to attend another mandatory meeting"
]

best_model = results[best_name]['model']
clean = [preprocess_text(h) for h in test_headlines]
vecs = tfidf.transform(clean)
preds = best_model.predict(vecs)

print('Custom Headline Predictions:')
print('-' * 70)
for h, p in zip(test_headlines, preds):
    label = '🎭 SARCASTIC' if p == 1 else '📰 NOT SARCASTIC'
    print(f'{label}: {h}')